In [1]:
""" Otto GNN Optimized v2 - Multi-Behavior LightGCN 
Improvements: Type embedding, Target-aware attention, T=3 layers 
"""

import numpy as np
import pandas as pd
import os, glob, gc, math, random
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.amp import autocast, GradScaler

import polars as pl
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import wandb

In [2]:
!pip install torch_geometric wandb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.2 MB/s eta 0:00:00


In [3]:
SEED = 1023
TYPE_WEIGHTS = {'clicks': 1.0, 'carts': 6.0, 'orders': 3.0}

class Config:
    seed = SEED
    embed_dim = 64
    num_layers = 3  # T=3 optimal (SR-GNN)
    batch_size = 4096
    epochs = 30
    learning_rate = 1e-3  # Higher than v1
    weight_decay = 1e-5
    session_sample_rate = 0.10
    validation_split = 0.1
    warmup_epochs = 2
    early_stopping_patience = 5
    type_weights = TYPE_WEIGHTS

config = Config()

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(config.seed)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


In [4]:
type_map = {'clicks': 0, 'carts': 1, 'orders': 2}
type_weight_map = {0: 1.0, 1: 6.0, 2: 3.0}

def process_data(df):
    return df.with_columns([
        pl.col('type').replace(type_map).cast(pl.Int8),
        (pl.col('ts')/1000).cast(pl.Int32),
        pl.col('aid').cast(pl.Int32)
    ])

print("Data loading ready")

Data loading ready


In [5]:
parquet_files = glob.glob('/kaggle/input/datasets/cdeotte/otto-validation/train_parquet/*.parquet')
test_files = glob.glob('/kaggle/input/datasets/cdeotte/otto-validation/test_parquet/*.parquet')
print(f"Found {len(parquet_files)} train, {len(test_files)} test files")

# Load all train files
dfs = []
for f in tqdm(parquet_files):
    try:
        df = pl.read_parquet(f)
        dfs.append(process_data(df))
    except: pass
full_train_df = pl.concat(dfs)
print(f"Loaded: {full_train_df.shape}")

Found 100 train, 20 test files


  0%|          | 0/100 [00:00<?, ?it/s]

Loaded: (163955180, 4)


In [6]:
print(f"Sampling {config.session_sample_rate*100}% sessions")
all_sessions = full_train_df['session'].unique()
sampled_sessions = all_sessions.sample(fraction=config.session_sample_rate, shuffle=True, seed=config.seed)
print(f"Sampled: {len(sampled_sessions)}/{len(all_sessions)}")
train_data = full_train_df.filter(pl.col('session').is_in(sampled_sessions))
print(f"Train data: {train_data.shape}")
del full_train_df; gc.collect()

Sampling 10.0% sessions
Sampled: 1109852/11098528


/tmp/ipykernel_43/327072673.py:5: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  train_data = full_train_df.filter(pl.col('session').is_in(sampled_sessions))


Train data: (16357578, 4)


16

In [7]:
print("Creating node mappings")
unique_sessions = train_data['session'].unique()
unique_aids = train_data['aid'].unique()
num_sessions = len(unique_sessions)
num_aids = len(unique_aids)
total_nodes = num_sessions + num_aids
print(f"Sessions: {num_sessions}, Items: {num_aids}, Total: {total_nodes}")

session2idx = {s: i for i, s in enumerate(unique_sessions)}
aid2idx = {a: i+num_sessions for i, a in enumerate(unique_aids)}
idx2aid = {i+num_sessions: a for i, a in enumerate(unique_aids)}

Creating node mappings
Sessions: 1109852, Items: 1189317, Total: 2299169


In [8]:
print("Building edge list")
interactions = train_data.select(['session','aid','type']).unique()
interactions = interactions.with_columns([
    pl.col('session').replace(session2idx),
    pl.col('aid').replace(aid2idx)
])
source_nodes = interactions['session'].to_numpy()
target_nodes = interactions['aid'].to_numpy()
edge_types = interactions['type'].to_numpy()
weights = np.array([type_weight_map[t] for t in edge_types])

edge_index = np.array([np.concatenate([source_nodes, target_nodes]), np.concatenate([target_nodes, source_nodes])])
edge_weights = np.concatenate([weights, weights])
print(f"Edges: {edge_index.shape[1]}")
del interactions; gc.collect()

Building edge list
Edges: 22978162


0

In [9]:
from torch_geometric.data import Data
edge_index_tensor = torch.LongTensor(edge_index).to(DEVICE)
edge_weight_tensor = torch.FloatTensor(edge_weights).to(DEVICE)
print("Graph data ready")

Graph data ready


In [10]:
train_df_indexed = train_data.with_columns([
    pl.col('session').replace(session2idx),
    pl.col('aid').replace(aid2idx)
])
user_pos_items = train_df_indexed.group_by('session').agg(pl.col('aid'))
user_pos_dict = {row[0]: set(row[1]) for row in user_pos_items.rows()}
interaction_pairs = list(train_df_indexed.select(['session','aid','type']).unique().rows())
train_pairs, val_pairs = train_test_split(interaction_pairs, test_size=config.validation_split, random_state=config.seed)
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")
all_item_indices = set(aid2idx.values())
del train_data, train_df_indexed; gc.collect()

Train: 10340172, Val: 1148909


0

In [11]:
class BPRDataset(Dataset):
    def __init__(self, pairs, all_items, user_pos):
        self.pairs, self.all_items, self.user_pos = pairs, list(all_items), user_pos
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        u, i, t = self.pairs[idx]
        neg = random.choice(self.all_items)
        while neg in self.user_pos.get(u, set()): neg = random.choice(self.all_items)
        return torch.tensor(u), torch.tensor(i), torch.tensor(neg), torch.tensor(t)

train_ds = BPRDataset(train_pairs, all_item_indices, user_pos_dict)
val_ds = BPRDataset(val_pairs, all_item_indices, user_pos_dict)
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, num_workers=4, pin_memory=True)
print(f"Loaders: Train={len(train_loader)}, Val={len(val_loader)}")

Loaders: Train=2525, Val=281


In [12]:
from torch_geometric.nn.conv import GCNConv

class MultiBehaviorLightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64, num_layers=3, dropout=0.1):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding = nn.Embedding(num_users + num_items, embed_dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        self.type_embedding = nn.Embedding(3, embed_dim)  # NEW: type embedding
        nn.init.xavier_uniform_(self.type_embedding.weight)
        self.convs = nn.ModuleList([GCNConv(embed_dim, embed_dim, cached=False) for _ in range(num_layers)])
        self.agg_weights = nn.Parameter(torch.ones(num_layers+1)/(num_layers+1))  # NEW: learnable weights
        self.dropout = nn.Dropout(dropout)
    def forward(self, edge_index, edge_weight=None):
        x = self.embedding.weight
        all_emb = [x]
        for conv in self.convs:
            x = conv(x, edge_index, edge_weight)
            x = self.dropout(x)
            all_emb.append(x)
        weights = F.softmax(self.agg_weights, dim=0)
        final_emb = sum(w*e for w,e in zip(weights, all_emb))
        return torch.split(final_emb, [self.num_users, self.num_items])

model = MultiBehaviorLightGCN(num_sessions, num_aids, config.embed_dim, config.num_layers).to(DEVICE)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 147,159,492


In [13]:
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
def get_lr(epoch):
    if epoch < config.warmup_epochs: return (epoch+1)/config.warmup_epochs
    progress = (epoch-config.warmup_epochs)/(config.epochs-config.warmup_epochs)
    return 0.5*(1+math.cos(math.pi*progress))
scheduler = optim.lr_scheduler.LambdaLR(optimizer, get_lr)
scaler = GradScaler()
print("Optimizer ready")

Optimizer ready


In [14]:
def bpr_loss(u_emb, pos_emb, neg_emb):
    return -torch.mean(F.logsigmoid(torch.sum(u_emb*pos_emb, dim=-1) - torch.sum(u_emb*neg_emb, dim=-1)))

def recall_at_k(users_emb, items_emb, user_pos, k=20):
    scores = []
    sample_users = random.sample(list(user_pos.keys()), min(1000, len(user_pos)))
    for u in sample_users:
        if u >= len(users_emb): continue
        s = torch.matmul(users_emb[u].unsqueeze(0), items_emb.T).squeeze()
        _, top = torch.topk(s, min(k, len(s)))
        top_items = set((top + num_sessions).cpu().tolist())
        pos = user_pos.get(u, set())
        if len(pos): scores.append(len(top_items & pos) / len(pos))
    return np.mean(scores) if scores else 0.0

def ndcg_at_k(users_emb, items_emb, user_pos, k=20):
    scores = []
    sample_users = random.sample(list(user_pos.keys()), min(500, len(user_pos)))
    for u in sample_users:
        if u >= len(users_emb): continue
        s = torch.matmul(users_emb[u].unsqueeze(0), items_emb.T).squeeze()
        _, top = torch.topk(s, min(k, len(s)))
        top_items = set((top + num_sessions).cpu().tolist())
        pos = user_pos.get(u, set())
        if not len(pos): continue
        dcg = sum(1/math.log2(i+2) for i,t in enumerate(top_items) if t in pos)
        idcg = sum(1/math.log2(i+2) for i in range(min(k, len(pos)))
        scores.append(dcg/idcg if idcg else 0)
    return np.mean(scores) if scores else 0.0

print("Metrics ready")


Metrics ready


In [15]:
try:
    wandb.init(project="otto-gnn-v2", config={k:getattr(config,k) for k in ['embed_dim','num_layers','batch_size','learning_rate']}, mode="disabled")
    USE_WANDB = True
except: USE_WANDB = False
print(f"WandB: {USE_WANDB}")

WandB: True


In [16]:
print("Starting training")
best_recall, best_state, patience = 0, None, 0

for epoch in tqdm(range(config.epochs)):
    model.train()
    train_loss = 0
    for u, pos, neg, t in train_loader:
        u, pos, neg, t = u.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE), t.to(DEVICE)
        optimizer.zero_grad()
        with autocast('cuda'):
            users_emb, items_emb = model(edge_index_tensor, edge_weight_tensor)
            type_emb = model.type_embedding(t)  # NEW: use type embedding
            loss = bpr_loss(users_emb[u], items_emb[pos-num_sessions]+type_emb, items_emb[neg-num_sessions])
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    scheduler.step()

    model.eval()
    users_emb, items_emb = model(edge_index_tensor, edge_weight_tensor)
    recall = recall_at_k(users_emb, items_emb, user_pos_dict)
    ndcg = ndcg_at_k(users_emb, items_emb, user_pos_dict)
    lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, recall={recall:.4f}, ndcg={ndcg:.4f}, lr={lr:.6f}")
    if USE_WANDB: wandb.log({'train_loss':train_loss, 'recall':recall, 'ndcg':ndcg})
    
    if recall > best_recall:
        best_recall, best_state, patience = recall, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        print(f"  ** Best! {best_recall:.4f}")
    else:
        patience += 1
        if patience >= config.early_stopping_patience:
            print("Early stopping"); break

print(f"Best Recall@20: {best_recall:.4f}")


Starting training


  0%|          | 0/30 [00:00<?, ?it/s]

TypeError: autocast.__init__() missing 1 required positional argument: 'device_type'

In [ ]:
if best_state:
    model.load_state_dict(best_state)
model.eval()
users_emb, items_emb = model(edge_index_tensor, edge_weight_tensor)
torch.save(users_emb.cpu(), '/kaggle/working/user_embeddings.pt')
torch.save(items_emb.cpu(), '/kaggle/working/item_embeddings.pt')
print("Embeddings saved")

In [ ]:
print("="*50)
print("SUMMARY")
print("="*50)
print(f"Best Recall@20: {best_recall:.4f}")
print(f"Model: Multi-Behavior LightGCN v2")
print("Optimizations vs v1:")
print("1. Type embedding (clicks/carts/orders)")
print("2. T=3 layers (SR-GNN optimal)")
print("3. Learnable aggregation weights")
print("4. LR=1e-3 (vs 5e-4)")
print("5. Early stopping")